# 2. Tratamento de Nulos e Padronização

**Decisões de limpeza:**
- `orders`: converter todas as colunas de data de `string` para `datetime`
- `orders`: filtrar apenas pedidos com status `delivered` (únicos com data de entrega real)
- `products`: preencher categorias sem nome com `'unknown'`
- `reviews`: os comentários nulos são esperados (campo opcional) — mantemos apenas `review_score`
- `order_items`: sem nulos, mas vamos agregar por `order_id` para facilitar o join

In [1]:
import pandas as pd

df_olist_customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
df_olist_geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")
olist_order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
olist_order_payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
olist_order_reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
olist_orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
olist_products = pd.read_csv("../data/raw/olist_products_dataset.csv")
olist_sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
product_category_name_translation = pd.read_csv("../data/raw/product_category_name_translation.csv") 

In [2]:
# ============================================================
# ORDERS — CONVERSÃO DE DATAS E FILTRO DE STATUS
# ============================================================
# As colunas de data chegam como string (object).
# Precisamos convertê-las para datetime para calcular diferenças de tempo (métricas de SLA).
#
# Filtramos apenas pedidos 'delivered' porque:
#   → São os únicos com data de entrega real ao cliente
#   → Permitem calcular se houve atraso
#   → Base para todas as métricas de SLA
# ============================================================

date_cols = [
    'order_purchase_timestamp',       # Quando o cliente realizou o pedido
    'order_approved_at',              # Quando o pagamento foi aprovado
    'order_delivered_carrier_date',   # Quando foi entregue à transportadora
    'order_delivered_customer_date',  # Quando chegou ao cliente (DATA REAL)
    'order_estimated_delivery_date'   # Prazo prometido ao cliente
]

# Converte cada coluna para datetime, erros viram NaT
for col in date_cols:
    olist_orders[col] = pd.to_datetime(olist_orders[col], errors='coerce')

# Filtra apenas pedidos entregues
df_orders_delivered = olist_orders[olist_orders['order_status'] == 'delivered'].copy()

print(f'Total de pedidos na base:          {len(olist_orders):,}')
print(f'Pedidos com status delivered:      {len(df_orders_delivered):,} ({len(df_orders_delivered)/len(olist_orders)*100:.1f}%)')
print(f'Pedidos com outros status (excl.): {len(olist_orders) - len(df_orders_delivered):,}')

Total de pedidos na base:          99,441
Pedidos com status delivered:      96,478 (97.0%)
Pedidos com outros status (excl.): 2,963


In [3]:
df_orders_delivered.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [4]:
df_orders_delivered.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [5]:
df_orders_delivered.to_pickle("../data/processed/df_orders_delivered.pkl")

In [6]:
# ============================================================
# PRODUCTS — TRADUÇÃO DE CATEGORIAS
# ============================================================
# Fazemos o join com a tabela de tradução para ter os nomes
# das categorias em inglês (padrão para análises).
# Categorias sem tradução recebem 'unknown'.
# ============================================================

df_products_clean = olist_products.merge(
    product_category_name_translation,             # Tabela de-para: PT → EN
    on='product_category_name',                    # Chave de join
    how='left'                                     # Left: mantém todos os produtos
)

In [7]:
# Preenche categorias sem tradução
df_products_clean['product_category_name_english'] = (
    df_products_clean['product_category_name_english'].fillna('unknown')
)
df_products_clean.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,housewares


In [8]:
df_products_clean[df_products_clean['product_category_name_english'] == 'unknown']


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0,unknown
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0,unknown
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0,unknown
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0,unknown
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0,unknown
...,...,...,...,...,...,...,...,...,...,...
32515,b0a0c5dd78e644373b199380612c350a,NaN,NaN,NaN,NaN,1800.0,30.0,20.0,70.0,unknown
32589,10dbe0fbaa2c505123c17fdc34a63c56,NaN,NaN,NaN,NaN,800.0,30.0,10.0,23.0,unknown
32616,bd2ada37b58ae94cc838b9c0569fecd8,NaN,NaN,NaN,NaN,200.0,21.0,8.0,16.0,unknown
32772,fa51e914046aab32764c41356b9d4ea4,NaN,NaN,NaN,NaN,1300.0,45.0,16.0,45.0,unknown


In [9]:
df_products_clean.to_pickle("../data/processed/df_products_clean.pkl")

In [10]:
# ============================================================
# REVIEWS — LIMPEZA E DEDUPLICAÇÃO
# ============================================================
# Mantemos apenas order_id e review_score.
# Os campos de comentário têm ~88% de nulos (campo opcional) e não são necessários para as métricas principais.
# Removemos duplicatas de order_id (mantemos a 1ª avaliação).
# ============================================================

df_reviews_duplicadas = olist_order_reviews[olist_order_reviews.duplicated(subset='order_id', keep=False)]
df_reviews_duplicadas

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
30,540e7bbb2d06cfb7f85f3a88ba7ac97f,cf73e2cb1f4a9480ed70c154da3d954a,5,NaN,NaN,2018-01-18 00:00:00,2018-01-18 19:12:30
344,a0a641414ff718ca079b3967ef5c2495,169d7e0fd71d624d306f132acd791cbe,5,NaN,NaN,2018-03-04 00:00:00,2018-03-06 20:12:53
498,505a882ba08a689682a4afc6eb4e5965,1c308eca3f339414a92e518e2a2e5ee9,2,NaN,NaN,2017-12-28 00:00:00,2017-12-31 20:25:02
764,c5976a5a98e854fb23d7e03c6754ae60,2002ea16e75277eaa0b5d78632048540,5,NaN,NaN,2017-08-08 00:00:00,2017-08-10 11:11:29
778,62c7722239b976d943ec0d430cfe890e,1d297b4800ed1a3c5b0944d84c01ee99,3,NaN,NaN,2017-10-22 00:00:00,2017-10-31 15:33:32
...,...,...,...,...,...,...,...
98989,dfb3db02188d809d5cd199496b6da87e,c0db7d31ace61fc360a3eaa34dd3457c,5,NaN,NaN,2018-02-17 00:00:00,2018-02-19 19:29:19
99108,2c6c08892b83ba4c1be33037c2842294,42ae1967f68c90bb325783ac55d761ce,4,NaN,"Chegou um pouco amassada, mas nada de mais, e ...",2017-07-03 00:00:00,2017-07-05 19:06:59
99164,2afe63a67dfd99b3038f568fb47ee761,c5334d330e36d2a810a7a13c72e135ee,5,NaN,"Muito bom, produto conforme anunciado, entrega...",2018-03-03 00:00:00,2018-03-04 22:56:47
99178,44d1e9165ec54b1d89d33594856af859,a7dbcf5043158d6fa72859eead2f3d10,4,NaN,NaN,2017-05-24 00:00:00,2017-05-24 23:15:21


In [11]:
olist_order_reviews['order_id'].value_counts()

order_id
c88b1d1b157a9999ce368f218a407141    3
df56136b8031ecd28e200bb18e6ddb2e    3
03c939fd7fd3b38f8485a0f95798f1f6    3
8e17072ec97ce29f0e1f111e598b0c85    3
cf73e2cb1f4a9480ed70c154da3d954a    2
                                   ..
2a8c23fee101d4d5662fa670396eb8da    1
22ec9f0669f784db00fa86d035cf8602    1
55d4004744368f5571d1f590031933e4    1
7725825d039fc1f0ceb7635e3f7d9206    1
90531360ecb1eec2a1fbb265a0db0508    1
Name: count, Length: 98673, dtype: int64

In [12]:
olist_order_reviews[olist_order_reviews['order_id'] == 'c88b1d1b157a9999ce368f218a407141']

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
1985,ffb8cff872a625632ac983eb1f88843c,c88b1d1b157a9999ce368f218a407141,3,NaN,NaN,2017-07-22 00:00:00,2017-07-26 13:41:07
82525,202b5f44d09cd3cfc0d6bd12f01b044c,c88b1d1b157a9999ce368f218a407141,5,NaN,NaN,2017-07-22 00:00:00,2017-07-26 13:40:22
89360,fb96ea2ef8cce1c888f4d45c8e22b793,c88b1d1b157a9999ce368f218a407141,5,NaN,NaN,2017-07-21 00:00:00,2017-07-26 13:45:15


In [13]:
# Mantemos a avaliação mais RECENTE por pedido
# Critério de ordenação:
#   1º review_creation_date desc  → data de criação mais recente
#   2º review_answer_timestamp desc → desempate: resposta mais recente
df_reviews_clean = (
    olist_order_reviews[['order_id', 'review_score', 'review_creation_date', 'review_answer_timestamp']]
    .sort_values(
        ['review_creation_date', 'review_answer_timestamp'],
        ascending=[False, False]  # ambos decrescentes → mais recente primeiro
    )
    .drop_duplicates(subset='order_id', keep='first')                      # mantém a mais recente
    .drop(columns=['review_creation_date', 'review_answer_timestamp'])     # remove colunas auxiliares
    .copy()
)

print(f"Total de avaliações originais : {len(olist_order_reviews):,}")
print(f"Após deduplicar por order_id  : {len(df_reviews_clean):,}")
df_reviews_clean.head()

Total de avaliações originais : 99,224
Após deduplicar por order_id  : 98,673


,order_id,review_score
46904,0b223d92c27432930dfe407c6aea3041,5
39127,e118373a9e97e99fd3876fe78d34f501,5
7667,52018484704db3661b98ce838612b507,1
43092,e98eaa3acbf55a63a501cec1baf45f9e,1
10675,c15790c4480e97b6a152024a4f53f0f6,5


In [14]:
df_reviews_clean[df_reviews_clean['order_id'] == 'c88b1d1b157a9999ce368f218a407141']    

,order_id,review_score
1985,c88b1d1b157a9999ce368f218a407141,3


In [15]:
df_reviews_clean.to_pickle("../data/processed/df_reviews_clean.pkl")

In [16]:
# ============================================================
# ORDER_ITEMS — AGREGAÇÃO POR PEDIDO
# ============================================================
# Um pedido pode ter múltiplos itens. Para o join com orders
# precisamos de 1 linha por pedido. Agregamos:
#   - total_price:   soma dos preços dos itens
#   - total_freight: soma dos fretes dos itens
#   - n_items:       quantidade de itens no pedido
#   - seller_id:     vendedor principal (primeiro item)
#   - product_id:    produto principal (primeiro item)
# ============================================================

df_items_agg = (
    olist_order_items
    .sort_values(['order_id', 'order_item_id'])   # garante ordem: item 1, 2, 3...
    .groupby('order_id')
    .agg(
        total_price    = ('price',         'sum'),
        total_freight  = ('freight_value', 'sum'),
        n_items        = ('order_item_id', 'count'),
        seller_id      = ('seller_id',     'first'),   # seller do item 1
        product_id     = ('product_id',    'first')    # produto do item 1
    )
    .reset_index()
)

print(f'Total de pedidos agregados: {len(df_items_agg):,}')
df_items_agg.head()

Total de pedidos agregados: 98,666


,order_id,total_price,total_freight,n_items,seller_id,product_id
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,1,48436dade18ac8b2bce089ec2a041202,4244733e06e7ecb4970a6e2683c13e61
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,1,dd7ddc04e1b6c2c614352b383efe2d36,e5f2d52b802189ee658865ca93d83a8f
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,1,5b51032eddd242adc84c38acab88f23d,c777355d18b72b67abbeef9df44fd0fd
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,1,9d7a1d34a5052409006425275ba1c2b4,7634da152a4610f1595efa32f14722fc
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,1,df560393f3a51e74553ab94004ba5c87,ac6c3623068f30de03045865e4e10089


In [17]:
df_items_agg.to_pickle("../data/processed/df_items_agg.pkl")

In [18]:
# ============================================================
# VALIDAÇÃO — pedido com múltiplos itens
# ============================================================
# 1. Buscamos um order_id que tenha mais de 1 item no original
# 2. Mostramos as linhas originais
# 3. Mostramos como ficou no df_items_agg (1 linha só)
# ============================================================

# Pega o primeiro order_id que tem mais de 1 item
exemplo_id = (
    olist_order_items
    .groupby('order_id')['order_item_id']
    .count()
    .pipe(lambda s: s[s > 1])
    .index[0]
)

print(f'order_id escolhido: {exemplo_id}\n')

print('--- ORIGINAL (olist_order_items) ---')
print(olist_order_items[olist_order_items['order_id'] == exemplo_id]
      [['order_id','order_item_id','product_id','seller_id','price','freight_value']])

print('\n--- AGREGADO (df_items_agg) ---')
print(df_items_agg[df_items_agg['order_id'] == exemplo_id])

order_id escolhido: 0008288aa423d2a3f00fcb17cd7d8719

--- ORIGINAL (olist_order_items) ---
                            order_id  order_item_id  \
13  0008288aa423d2a3f00fcb17cd7d8719              1   
14  0008288aa423d2a3f00fcb17cd7d8719              2   

                          product_id                         seller_id  price  \
13  368c6c730842d78016ad823897a372db  1f50f920176fa81dab994f9023523100   49.9   
14  368c6c730842d78016ad823897a372db  1f50f920176fa81dab994f9023523100   49.9   

    freight_value  
13          13.37  
14          13.37  

--- AGREGADO (df_items_agg) ---
                            order_id  total_price  total_freight  n_items  \
13  0008288aa423d2a3f00fcb17cd7d8719         99.8          26.74        2   

                           seller_id                        product_id  
13  1f50f920176fa81dab994f9023523100  368c6c730842d78016ad823897a372db  


In [19]:
exemplo_id = (
    olist_order_items
    .groupby('order_id')['seller_id']
    .nunique()
    .pipe(lambda s: s[s > 1])
    .index[0]
)

original = (
    olist_order_items[olist_order_items['order_id'] == exemplo_id]
    [['order_item_id','product_id','seller_id','price','freight_value']]
    .sort_values('order_item_id')
)
print('--- ORIGINAL ---')
print(original)
print('\n--- AGREGADO ---')
print(df_items_agg[df_items_agg['order_id'] == exemplo_id])

--- ORIGINAL ---
    order_item_id                        product_id  \
80              1  d41dc2f2979f52d75d78714b378d4068   
81              2  880be32f4db1d9f6e2bec38fb6ac23ab   

                           seller_id  price  freight_value  
80  7299e27ed73d2ad986de7f7c77d919fa   8.99          32.57  
81  fa40cc5b934574b62717c68f3d678b6d  44.90           7.16  

--- AGREGADO ---
                            order_id  total_price  total_freight  n_items  \
73  002f98c0f7efd42638ed6100ca699b42        53.89          39.73        2   

                           seller_id                        product_id  
73  7299e27ed73d2ad986de7f7c77d919fa  d41dc2f2979f52d75d78714b378d4068  
